# ESCM — behavioral parity audit

Compare an **original-code reference** against an **independently constructed network**, using identical recorded spikes, neuron numbering, geometry, parameters, and simulation clocks. This is a regression test, **not** a test of the paper's mathematical correctness.

The legacy model has unusual Brian2 scheduling and an ambiguous `incoming_spikes_post` reset. We preserve the original equations and event rules. One necessary adjustment is explicit: the original `incoming_spikes_post` reset references a synaptic variable suffix inside a neuron reset. Both networks instead reset `incoming_spikes`. Consequently, this tests parity under that shared correction, **not byte-for-byte behavior of the unmodified script**.

The test uses a **40 × 30 crop** from the 640 × 480 Directions recording. With 1,200 neurons, the original 1,000-neuron connectivity chunk boundary is exercised. It is therefore much cheaper than the full camera but can reveal the chunk omission.

## 1. Recorded input, crop, and clock

Use the same decoded DVS events for both networks. Choose a busy crop automatically from the first `recording_window_s` seconds. Timestamp scaling and conversion to fixed 0.5 ms ticks are performed **once**, before either network is created. All pixel arithmetic uses int64 to avoid the previously discovered overflow.

This fixes the input sequence for the comparison, even though the legacy `BrianHF.event_to_spike` had a different deduplication procedure. We audit that preprocessing separately, rather than letting it confound the neural comparison.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from brian2 import *
from escm.event_input import load_events

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
# Legacy NeuronEquations.py sits at the repository root, not in escm.
sys.path.insert(0, str(root))
recording = load_events(
    root / "InputData/h36m_sample/cam2_S1_Directions/ch0dvs",
    channel="left", width=640, height=480, timestamp_clock_ns=1000,
)
data = recording["events"]
sensor_w, sensor_h = recording["width"], recording["height"]
crop_w, crop_h, chunk_size = 40, 30, 1000
clock_s = 0.0005
time_scale = 0.1
recording_window_s = 0.5   # unscaled recorded seconds
silent_s = 0.05

ts = np.asarray(data["ts"], dtype=np.float64)
x = np.asarray(data["x"], dtype=np.int64)
y = np.asarray(data["y"], dtype=np.int64)
assert len(ts) and len(ts) == len(x) == len(y)
near = (ts - ts[0]) < recording_window_s
# Choose a populated 40x30 region without looking at model outputs.
counts = np.zeros((sensor_h, sensor_w), dtype=np.int32)
np.add.at(counts, (y[near], x[near]), 1)
# Find busiest non-overlapping 40x30 tile, avoiding extra dependencies.
tiles = [(counts[yy:yy+crop_h, xx:xx+crop_w].sum(), xx, yy)
         for yy in range(0, sensor_h-crop_h+1, crop_h)
         for xx in range(0, sensor_w-crop_w+1, crop_w)]
_, x0, y0 = max(tiles)
inside = near & (x >= x0) & (x < x0+crop_w) & (y >= y0) & (y < y0+crop_h)
t = (ts[inside] - ts[0]) * time_scale
indices = (y[inside]-y0)*crop_w + (x[inside]-x0)
ticks = np.floor(t / clock_s).astype(np.int64)
pairs = np.unique(np.column_stack((ticks, indices)), axis=0)
assert len(pairs), "Chosen crop contains no events"
assert pairs[:, 1].min() >= 0 and pairs[:, 1].max() < crop_w*crop_h
input_duration = (int(pairs[:, 0].max())+1)*clock_s
run_duration = input_duration + silent_s
print(f"ROI x={x0}:{x0+crop_w} y={y0}:{y0+crop_h} | {len(pairs):,} mapped input spikes")
print(f"Stimulus={input_duration:.4f} s, silent extension={silent_s:.4f} s")
print("Input Y range:", (pairs[:, 1]//crop_w).min(), (pairs[:, 1]//crop_w).max())

## 2. Compare connectivity before simulating

The reference implements `BrianHF.calculate_Neighbours`' **1,000-index chunks** exactly: it only compares neurons within the same chunk. The candidate independently builds the complete geometric neighborhood, then retains same-chunk edges for this **parity mode**. We also report how many edges a corrected full-neighborhood implementation would add.

Both networks exclude explicit autapses; the old code updates `Exc_pre` on outgoing recurrent spikes instead.

In [ ]:
N = crop_w * crop_h
k = 8

def legacy_edges():
    pre, post = [], []
    for start in range(0, N, chunk_size):
        stop = min(start + chunk_size, N)
        ids = np.arange(start, stop)
        xx, yy = ids % crop_w, ids // crop_w
        matches = (np.abs(xx[:, None]-xx[None, :]) <= k) & (
            np.abs(yy[:, None]-yy[None, :]) <= k)
        np.fill_diagonal(matches, False)
        a, b = np.nonzero(matches)
        pre.append(ids[a]); post.append(ids[b])
    return np.concatenate(pre), np.concatenate(post)

def candidate_edges():
    pre, post = [], []
    for dy in range(-k, k+1):
        for dx in range(-k, k+1):
            if dx == 0 and dy == 0:
                continue
            xx = np.arange(max(0, -dx), min(crop_w, crop_w-dx))
            yy = np.arange(max(0, -dy), min(crop_h, crop_h-dy))
            if not len(xx) or not len(yy):
                continue
            xx, yy = np.meshgrid(xx, yy)
            a = (yy*crop_w + xx).ravel()
            b = a + dy*crop_w + dx
            pre.append(a); post.append(b)
    a, b = np.concatenate(pre), np.concatenate(post)
    return a, b, (a//chunk_size) == (b//chunk_size)

ref_pre, ref_post = legacy_edges()
all_pre, all_post, same_chunk = candidate_edges()
new_pre, new_post = all_pre[same_chunk], all_post[same_chunk]
ref_set = set(zip(ref_pre.tolist(), ref_post.tolist()))
new_set = set(zip(new_pre.tolist(), new_post.tolist()))
assert ref_set == new_set, "Connection mapping differs"
print("Legacy edges:", len(ref_set))
print("Corrected full-field edges:", len(all_pre))
print("Cross-chunk edges omitted by legacy:", int((~same_chunk).sum()))
print("Connectivity equivalence: PASS")

## 3. Construct two separate networks and record internal state

The reference equation comes from the legacy `NeuronEquations.py`. The candidate writes the equivalent four differential equations directly. Both preserve `ExtIn = beta`, two recurrent on-pre pathways, the legacy counter-reset *intent*, and the original `P_ON/P_OFF` event schedules. The original reset string uses `incoming_spikes_post` in a `NeuronGroup` reset, where this synaptic suffix is not valid. We use `incoming_spikes = 0` **in both networks** so the comparison can execute. This is a documented correction to the legacy source, not a claim of literal execution parity. This notebook tests whether the two implementations behave the same **under the same corrected reset**.

Record selected neurons' `v, ExtIn, Exc, Inh, P, incoming_spikes` at the exact same clock resolution. Compare raw spike indices and ticks, not videos.

In [ ]:
from NeuronEquations import EQ_SCM_IF

parameters = dict(tau=0.2*ms, vt=0.1, vr=0.0, beta=0.5, Wi=6.0, Wk=-3.0)
candidate_equations = """
dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau : 1
dExtIn/dt = -ExtIn/tau : 1
dExc/dt = -Exc/tau : 1
dInh/dt = -Inh/tau : 1
incoming_spikes : integer
P : integer
X : integer
Y : integer
"""
state_names = ["v", "ExtIn", "Exc", "Inh", "P", "incoming_spikes"]
tracked = np.unique(np.r_[pairs[:min(12, len(pairs)), 1], [0, 999, 1000, N-1]]).astype(int)

def evaluate(label, equations, pre_idx, post_idx):
    start_scope()
    defaultclock.dt = clock_s*second
    beta, Wi, Wk = parameters["beta"], parameters["Wi"], parameters["Wk"]
    tau, vt, vr = parameters["tau"], parameters["vt"], parameters["vr"]
    camera = SpikeGeneratorGroup(N, pairs[:, 1], pairs[:, 0]*defaultclock.dt, sorted=True)
    neurons = NeuronGroup(
        N, equations, threshold="v>vt",
        reset="v = vr; incoming_spikes = 0",
        refractory=0*ms,
        events={"P_ON": "v > vt",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
        method="exact",
    )
    neurons.X = np.arange(N) % crop_w
    neurons.Y = np.arange(N) // crop_w
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")
    feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta")
    feed.connect(j="i")
    recurrent = Synapses(neurons, neurons, on_pre={
        "pre": "incoming_spikes_post += 1; Exc_pre = Wi",
        "pre_2": "Inh_post = P_post * Wk * incoming_spikes_post",
    })
    recurrent.connect(i=pre_idx, j=post_idx)
    spikes = SpikeMonitor(neurons)
    states = StateMonitor(neurons, state_names, record=tracked, dt=defaultclock.dt)
    net = Network(camera, neurons, feed, recurrent, spikes, states)
    print(label, "running")
    net.run(run_duration*second)
    return {
        "spike_ticks": np.rint(np.asarray(spikes.t/second)/clock_s).astype(np.int64),
        "spike_ids": np.asarray(spikes.i, dtype=np.int64).copy(),
        "state_time": np.asarray(states.t/second).copy(),
        "state": {key: np.asarray(getattr(states, key)).copy() for key in state_names},
    }

# A compilation/scheduling error here is a genuine blocker in the legacy model.
reference = evaluate("Legacy reference", EQ_SCM_IF, ref_pre, ref_post)
candidate = evaluate("Candidate", candidate_equations, new_pre, new_post)

## 4. Exact parity report and first mismatch

Spikes must match **by neuron ID and simulation tick**. Integer states must match exactly; floating-state comparisons use a tight absolute tolerance. No claim of equivalence is made unless all assertions pass. If there is a failure, the report identifies the earliest mismatch and variable.

A passing result only proves equivalence for this crop, input segment, Brian2 version and execution backend. It does **not** validate full-resolution or corrected-connectivity behavior.

In [ ]:
r, c = reference, candidate
a = np.column_stack((r["spike_ticks"], r["spike_ids"]))
b = np.column_stack((c["spike_ticks"], c["spike_ids"]))
spike_equal = np.array_equal(a, b)
print(f"Reference spikes: {len(a):,}; candidate spikes: {len(b):,}")
print("Spike-train parity:", "PASS" if spike_equal else "FAIL")
if not spike_equal:
    first = next((i for i in range(min(len(a), len(b))) if not np.array_equal(a[i], b[i])),
                 min(len(a), len(b)))
    print("First differing spike index:", first)
    print("Reference:", a[first] if first < len(a) else "<end>")
    print("Candidate:", b[first] if first < len(b) else "<end>")

assert np.array_equal(r["state_time"], c["state_time"]), "Monitor clocks differ"
all_ok = spike_equal
for key in state_names:
    left, right = r["state"][key], c["state"][key]
    exact = key in ("P", "incoming_spikes")
    ok = np.array_equal(left, right) if exact else np.allclose(
        left, right, atol=1e-10, rtol=1e-10)
    diff = np.abs(left.astype(float) - right.astype(float))
    print(f"{key:17s} {'PASS' if ok else 'FAIL'} | max difference={diff.max():.3g}")
    if not ok:
        row, col = np.argwhere(diff > (0 if exact else 1e-10 + 1e-10*np.abs(left)))[0]
        print("  First mismatch: tracked neuron", int(tracked[row]),
              "at t=", float(r["state_time"][col]), "s")
    all_ok &= ok
print("OVERALL:", "PASS" if all_ok else "FAIL")
assert all_ok, "Behavioral parity failed; see first discrepancies above"

## Interpretation

If construction fails, record the exception. In particular, differences in Brian2 event scheduling between versions can prevent the original event handlers from running. Do not interpret a compilation failure as a behavioral mismatch. If parity passes, the next audit should vary the crop, timestamp sampling, and backend and then compare the full corrected-connectivity model. The input preparation and video rendering still require their **own** tests; this notebook intentionally holds input fixed.